# 25 — Error Handling in NLP
**Goal:** Build robust NLP pipelines that handle real-world messy data.

## 1. Common Resume Parsing Failures

In [ ]:
error_cases = [
    ("Empty document", ""),
    ("Binary file", "\x00\x01\x02\x03"),
    ("Encoding issue", "Jos\xe9's r\xe9sum\xe9 — Sr. ML Engr."),
    ("All caps", "PROFESSIONAL SUMMARY: DATA SCIENTIST"),
    ("No sections", "Just a wall of text with no breaks"),
    ("Special chars only", "!!!@@@###$$$%%%%"),
]

print("Common error cases in resume parsing:")
for case_name, text in error_cases:
    print(f"  {case_name:25s} → {repr(text[:30])}...")

## 2. Safe Encoding Handler

In [ ]:
import chardet

def safe_decode(data: bytes) -> str:
    """Handle encoding detection and fallback."""
    if not data:
        return ""
    try:
        # Try UTF-8 first
        return data.decode("utf-8")
    except UnicodeDecodeError:
        # Detect encoding
        detected = chardet.detect(data)
        enc = detected.get("encoding", "latin-1")
        print(f"  Warning: UTF-8 failed, detected {enc} (confidence: {detected['confidence']})")
        try:
            return data.decode(enc)
        except:
            # Fallback: ignore errors
            return data.decode("utf-8", errors="ignore")

# Test
test_data = b"Jos\xe9's r\xe9sum\xe9"
print(f"Raw bytes: {test_data}")
print(f"Decoded:   {safe_decode(test_data)}")

## 3. Graceful NLP Pipeline with Fallbacks

In [ ]:
import spacy, re
from typing import Optional, List

class RobustNLPPipeline:
    """NLP pipeline with fallback for every failure mode."""

    def __init__(self):
        try:
            self.nlp = spacy.load("en_core_web_sm")
            self.spacy_ok = True
        except OSError:
            print("Warning: spaCy model not available, using regex-only fallback")
            self.spacy_ok = False

    def extract_entities(self, text: str) -> List[tuple]:
        """Extract entities with fallback."""
        if not text or not text.strip():
            return []

        entities = []

        # Try spaCy first
        if self.spacy_ok:
            try:
                doc = self.nlp(text[:100000])  # Limit to avoid OOM
                entities = [(e.text, e.label_) for e in doc.ents]
            except Exception as e:
                print(f"  Warning: spaCy failed ({e}), falling back to regex")
                entities = []

        # Regex fallback for emails, phones, URLs
        if not entities:
            for pattern, label in [
                (r"[\w.+-]+@[\w-]+\.[\w.]+", "EMAIL"),
                (r"https?://[\w./-]+", "URL"),
                (r"[+]?[\d\s()-]{7,}[\d]", "PHONE"),
            ]:
                for match in re.finditer(pattern, text, re.IGNORECASE):
                    entities.append((match.group(0), label))

        return entities

    def __call__(self, text: str):
        return self.extract_entities(text)

# Test
robust = RobustNLPPipeline()
test_cases = [
    "Contact: john@email.com, Phone: +1-555-1234",
    "",
    "\x00\x01\x02Binary garbage\x03",
]

for tc in test_cases:
    result = robust(tc)
    print(f"Input: {repr(tc[:40]):42s} → Entities: {result}")

## 4. Validation Layer for Extracted Data

In [ ]:
import re
from typing import Optional

def validate_email(email: str) -> Optional[str]:
    """Validate and normalize email."""
    if not email: return None
    email = email.strip().lower()
    pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
    return email if re.match(pattern, email) else None

def validate_phone(phone: str) -> Optional[str]:
    """Validate and normalize phone."""
    if not phone: return None
    digits = re.sub(r"[^\d+]", "", phone)
    return digits if 7 <= len(digits) <= 15 else None

def validate_year(year: str) -> Optional[int]:
    """Validate year is reasonable."""
    try:
        y = int(year.strip())
        return y if 1950 <= y <= 2030 else None
    except: return None

# Test
tests = [
    ("john@email.com", "john@email.com"),
    ("not-an-email", None),
    ("+1-555-123-4567", "+15551234567"),
    ("12", None),
    ("2020", 2020),
]
for raw, expected in tests:
    for validator, name in [(validate_email, "email"), (validate_phone, "phone"), (validate_year, "year")]:
        result = validator(raw)
        if result is not None:
            print(f"  {name:7s} '{raw:20s}' → valid: {result}")
            break
    else:
        print(f"  {'?':7s} '{raw:20s}' → invalid (expected {expected})")

## 5. Empty Section Detection

In [ ]:
def detect_empty_sections(text: str) -> List[str]:
    """Find resume sections that exist but have no substantive content."""
    sections = re.split(r"\n(?=[A-Z][A-Za-z /]+\n)", text)
    empty = []
    for section in sections:
        lines = [l.strip() for l in section.split("\n") if l.strip()]
        header = lines[0] if lines else ""
        content_lines = [l for l in lines[1:] if len(l) > 10]
        if header and not content_lines:
            empty.append(header)
    return empty

resume = """Professional Summary
Experienced data scientist.

Skills

Experience
Developed ML models at Google.

Education

Certifications

"""
empty = detect_empty_sections(resume)
print("Empty sections found:")
for section in empty:
    print(f"  ⚠ '{section}' has no content")

## Summary: Production NLP needs error handling at every stage. Defensive coding prevents pipeline failures.